# 05 · Predict and write the submission

Same idea as the organisers' notebook 02:
1. Train the chosen setting on **all training recordings** (all 6 subjects).
2. Save the trained models to `weights/` (sharing weights is required).
3. **Format check on sub-01**: write a ranked list exactly like the real submission and score
   it with sub-01's answers. If the number matches notebook 04, the format is right.
4. Write `submission/sub-05_ranked_candidate_list.npy` and `sub-06_…`: 102 × 102 integers.
   Row *i* = the *i*-th recording of `tst/sub-0X_tst.npy` in its **original order**. Each row
   holds the video ids 1001–1102, most likely first.

The sub-05/06 test recordings are used **only here**, once, to produce the predictions.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root, so `mindreading` imports work
import numpy as np
import matplotlib.pyplot as plt
from mindreading.config import DATA_DIR, RESULTS_DIR, WEIGHTS_DIR, SUBMISSION_DIR
print('data folder:', DATA_DIR)

data folder: C:\Users\canbe\Desktop\eeg-mind-reading-challenge\eeg-mind-reading-challenge


In [2]:
import joblib
from mindreading.pipeline import Config, Pipeline
from mindreading.video_features import extract_clip
from mindreading.config import TEST_IDS
from shared.ranking import scores_to_ranked_ids, check_ranked, evaluate, save_submission

cfg = Config(**json.loads((RESULTS_DIR / 'chosen_config.json').read_text()))
feats = extract_clip()
pipe = Pipeline(cfg, feats)
models = pipe.fit(None) if cfg.pool_subjects else {s: pipe.fit(s) for s in range(1, 7)}
WEIGHTS_DIR.mkdir(exist_ok=True)
joblib.dump({'config': cfg.__dict__, 'models': models}, WEIGHTS_DIR / 'base_ridge.joblib', compress=3)
print('saved', WEIGHTS_DIR / 'base_ridge.joblib')

saved C:\Users\canbe\Desktop\Projects\Hackathon\hackathon_mind_reading\weights\base_ridge.joblib


In [3]:
def ranked_for(sub):
    eeg, labels = pipe.test_data(sub)                  # original shuffled order, never re-sorted
    m = models if cfg.pool_subjects else models[sub]
    scores = pipe.scores(m, eeg, TEST_IDS)['combined' if len(cfg.methods) > 1 else cfg.methods[0]]
    return check_ranked(scores_to_ranked_ids(scores)), labels

r1, labels1 = ranked_for(1)
print('format check on sub-01:', r1.shape, r1.dtype, 'first row starts', r1[0, :5])
print('score with sub-01 answers:', evaluate(r1, labels1))

format check on sub-01: (102, 102) int64 first row starts [1035 1023 1075 1025 1077]
score with sub-01 answers: {'top1': 0.08823529411764706, 'top5': 0.3431372549019608}


In [4]:
SUBMISSION_DIR.mkdir(exist_ok=True)
for sub in (5, 6):
    ranked, _ = ranked_for(sub)
    path = save_submission(ranked, f'sub-{sub:02d}', SUBMISSION_DIR)
    back = check_ranked(np.load(path))
    print(path, back.shape, back.dtype, 'row 0 top-5:', back[0, :5])

C:\Users\canbe\Desktop\Projects\Hackathon\hackathon_mind_reading\submission/sub-05_ranked_candidate_list.npy (102, 102) int64 row 0 top-5: [1082 1087 1059 1053 1072]


C:\Users\canbe\Desktop\Projects\Hackathon\hackathon_mind_reading\submission/sub-06_ranked_candidate_list.npy (102, 102) int64 row 0 top-5: [1002 1034 1080 1014 1022]
